# semantic_search/04a — Cancer-type baseline

Trains only the non-text reference space, `cancer_type_baseline`: one-hot cancer type and nothing else, through the same nested-CV XGBoost as `04_predict`. It answers how much of each target is explained by diagnosis alone — an embedding space that does not beat it has not shown it reads anything beyond cancer type.

**Runs after** `04_predict`. For each target × window the baseline is trained on exactly the patients the existing `concat` model used, read from that run's out-of-fold predictions (`--cohort-from concat`). `concat` itself is not loaded, checked, or retrained, and `OVERWRITE` only ever touches baseline artifacts. Targets with no `concat` predictions in a window are skipped.

The `cancer_type` target is excluded: the baseline features are that target's own labels.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path

from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "semantic_search").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402
from semantic_search import common  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<30} {path}")
    message = "All inputs present." if not missing else f"{len(missing)} missing: {', '.join(missing)}"
    print(f"\n{message}")
    return missing


def run_module(module: str, args: list[str] | None = None) -> int:
    """Run a module as a subprocess and stream its output."""
    cmd = [sys.executable, "-m", module] + (args or [])
    print("$ " + " ".join(cmd) + "\n", flush=True)
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"\nexit={proc.returncode}  elapsed={(time.time() - t0) / 60:,.1f} min", flush=True)
    return proc.returncode


print(f"repo root:  {REPO_ROOT}")
print(f"data root:  {config.DATA_PATH}")
print(f"this arm:   {config.SEMANTIC_SEARCH_PATH}")

## Configuration

Keep `SEED` and the fold counts equal to `04_predict`'s: on the same patients and labels, that reproduces the same outer folds, so the two models' out-of-fold predictions are paired fold by fold.

In [ ]:
MODULE = "semantic_search.train_prediction_models"

from semantic_search.prediction_targets import TARGETS as ALL_TARGETS  # noqa: E402

TARGETS = [t for t in ALL_TARGETS if t != "cancer_type"]
BASELINE = common.BASELINE_SPACE
COHORT_FROM = common.FULL_SPACE_SOURCE  # "concat"
WINDOWS = common.DEFAULT_WINDOWS
MODELS = ["xgboost"]

TREATMENT_GRANULARITY = "category"
MIN_TREATMENT_CLASS_N = 25
OUTER_FOLDS = 5
INNER_FOLDS = 3
SEED = 1234
N_JOBS = -1
AVPC_NEPC_LABELS = config.AVPC_NEPC_LABELS_PATH
OVERWRITE = False  # replaces baseline artifacts only
RUN_MODELS = True

print(f"targets:  {TARGETS}")
print(f"windows:  {WINDOWS}")
print(f"baseline: {BASELINE}  (cohort from {COHORT_FROM} predictions)")
print(f"CV:       {OUTER_FOLDS} outer x {INNER_FOLDS} inner; n_jobs={N_JOBS} ({os.cpu_count()} CPUs visible)")

## Preconditions

The baseline is built in memory from `cancer_type_df.csv.gz`; the cohort comes from the `concat` prediction files listed below.

In [ ]:
def artifact_paths(target: str, space: str, window: str, model: str) -> list[str]:
    stem = f"{target}__{space}__{window}__{model}"
    return [
        os.path.join(common.PREDICTIONS_DIR, f"{stem}.parquet"),
        os.path.join(common.MODELS_DIR, f"{stem}.joblib"),
        os.path.join(common.PREDICTION_META_DIR, f"{stem}.json"),
    ]


missing = check_inputs(
    [("cancer-type labels", os.path.join(config.FEATURE_PATH, "cancer_type_df.csv.gz"))]
)
print()
for window in WINDOWS:
    have = [t for t in TARGETS if os.path.exists(artifact_paths(t, COHORT_FROM, window, MODELS[0])[0])]
    done = [t for t in have if all(os.path.exists(p) for p in artifact_paths(t, BASELINE, window, MODELS[0]))]
    print(f"[{window}] {len(have)} / {len(TARGETS)} targets have {COHORT_FROM} predictions; "
          f"{len(done)} already have baseline artifacts")
    skipped = [t for t in TARGETS if t not in have]
    if skipped:
        print(f"  will skip (run 04_predict first): {skipped}")

## Run baseline

Watch for `WARNING` lines: they mean some `concat` patients have no cancer type (so the baseline cohort is slightly smaller), or that a patient's current label differs from the one `concat` was trained on.

In [ ]:
args = [
    "--targets", *TARGETS,
    "--spaces", BASELINE,
    "--cohort-from", COHORT_FROM,
    "--windows", *WINDOWS,
    "--models", *MODELS,
    "--treatment-granularity", TREATMENT_GRANULARITY,
    "--min-treatment-class-n", str(MIN_TREATMENT_CLASS_N),
    "--outer-folds", str(OUTER_FOLDS),
    "--inner-folds", str(INNER_FOLDS),
    "--seed", str(SEED),
    "--n-jobs", str(N_JOBS),
    "--avpc-nepc-labels", str(AVPC_NEPC_LABELS),
]
if OVERWRITE:
    args.append("--overwrite")

if RUN_MODELS:
    rc = run_module(MODULE, args)
    if rc != 0:
        print("\nTraining failed — see the traceback above. Completed setups remain reusable.")
else:
    print("RUN_MODELS = False; skipped training and continuing to existing results.")

## Baseline vs concat

Pooled out-of-fold metrics. `delta` is `concat − baseline`: how much the embeddings add beyond cancer type. `n_patients` should match between the two; a small shortfall on the baseline side is `concat` patients with no cancer type.

In [ ]:
import polars as pl

summary_path = common.result_path("prediction_metrics_summary")
if os.path.exists(summary_path):
    summary = pl.read_csv(summary_path).filter(
        pl.col("target").is_in(TARGETS)
        & pl.col("space").is_in([COHORT_FROM, BASELINE])
        & pl.col("window").is_in(WINDOWS)
        & pl.col("model").is_in(MODELS)
    )
    keys = ["target", "window", "model"]
    for metric in ["n_patients", "macro_ovr_auc_pooled_oof", "macro_f1_pooled_oof"]:
        wide = (
            summary.select(*keys, "space", metric)
            .pivot(on="space", index=keys, values=metric)
        )
        if {COHORT_FROM, BASELINE} <= set(wide.columns):
            wide = wide.with_columns(
                (pl.col(COHORT_FROM) - pl.col(BASELINE)).alias("delta")
            ).sort(keys)
        print(metric)
        display(wide)
else:
    print(f"No prediction summary at {summary_path}")